# Семинар 8. Transformer OCR

https://huggingface.co/docs/transformers/model_doc/trocr 

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import requests
from PIL import Image

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

# load image from the IAM dataset
url = "https://fki.tic.heia-fr.ch/static/img/a01-122-02.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")

pixel_values = processor(image, return_tensors="pt").pixel_values
generated_ids = model.generate(pixel_values)

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(generated_text)


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

C:\Users\selin\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\selin\.cache\huggingface\hub\models--microsoft--trocr-base-handwritten. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

## Задание 1. Запустите распознавание собственного документа и сравните результат с easyocr из 7 лабораторной. 

In [ ]:
# Решение задания: распознавание собственного документа и сравнение с EasyOCR
import re
import unicodedata
import numpy as np
import easyocr

document_path = "data/document.jpg"
document_image = Image.open(document_path).convert("RGB")

pixel_values = processor(document_image, return_tensors="pt").pixel_values
generated_ids = model.generate(pixel_values, max_new_tokens=64)
trocr_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

easyocr_reader = easyocr.Reader(['ru', 'en'], gpu=False)
easyocr_result = easyocr_reader.readtext(document_path)
easyocr_text = "\n".join(item[1] for item in easyocr_result)

expected_document_text = """
ДОГОВОР №123
г. Москва 01.01.2024
ООО "Технолоджи" в лице директора Иванова И.И.
и ООО "Поставщик" в лице директора Петрова П.П.
заключили настоящий договор о нижеследующем:
1. ПРЕДМЕТ ДОГОВОРА
1.1. Поставщик обязуется поставить товар.
1.2. Сумма договора: 1 500 000 рублей.
Подписи сторон:
Иванов И.И.
Петров П.П.
""".strip()

def normalize_ocr_text(text):
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def levenshtein_distance(a, b):
    a, b = normalize_ocr_text(a), normalize_ocr_text(b)
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(
                current[j - 1] + 1,
                previous[j] + 1,
                previous[j - 1] + (ca != cb),
            ))
        previous = current
    return previous[-1]

def character_accuracy(predicted, expected):
    expected_norm = normalize_ocr_text(expected)
    if not expected_norm:
        return 1.0
    return max(0.0, 1.0 - levenshtein_distance(predicted, expected) / len(expected_norm))

trocr_accuracy = character_accuracy(trocr_text, expected_document_text)
easyocr_accuracy = character_accuracy(easyocr_text, expected_document_text)

print("TrOCR:")
print(trocr_text)
print(f"Посимвольная точность TrOCR: {trocr_accuracy:.2%}")

print("\nEasyOCR:")
print(easyocr_text)
print(f"Посимвольная точность EasyOCR: {easyocr_accuracy:.2%}")

if easyocr_accuracy > trocr_accuracy:
    print("\nВывод: EasyOCR лучше подходит для этого русского печатного документа. Базовая TrOCR-модель из примера обучена преимущественно на англоязычном рукописном/печатном тексте и не решает задачу русскоязычного OCR без дообучения.")
else:
    print("\nВывод: TrOCR показал сопоставимое или лучшее качество на выбранном документе, но для устойчивого русскоязычного OCR все равно стоит проверять модель на большем наборе документов.")
